In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
from pyspark.sql.functions import current_timestamp

paths = [
    "/Volumes/nyctaxi_workspace/00_landing/data_sources/nyctaxi_yellow/2025-12/*.parquet",
    "/Volumes/nyctaxi_workspace/00_landing/data_sources/nyctaxi_yellow/2026-01/*.parquet",
    "/Volumes/nyctaxi_workspace/00_landing/data_sources/nyctaxi_yellow/2026-02/*.parquet",
    "/Volumes/nyctaxi_workspace/00_landing/data_sources/nyctaxi_yellow/2026-03/*.parquet",
    "/Volumes/nyctaxi_workspace/00_landing/data_sources/nyctaxi_yellow/2026-04/*.parquet"
]

df = spark.read.format("parquet").load(paths)

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "processed_timestamp",
    current_timestamp()
)

In [0]:
spark.sql("USE CATALOG nyctaxi_workspace")
spark.sql("USE SCHEMA nyctaxi_01_bronze")

df.write \
    .mode("overwrite") \
    .saveAsTable("yellow_trips_raw")

In [0]:
spark.sql("""
SELECT
    DATE_FORMAT(tpep_pickup_datetime, 'yyyy-MM') AS pickup_month,
    COUNT(*) AS total_rows
FROM nyctaxi_workspace.nyctaxi_01_bronze.yellow_trips_raw
GROUP BY DATE_FORMAT(tpep_pickup_datetime, 'yyyy-MM')
ORDER BY pickup_month
""").display()

pickup_month,total_rows
2001-01,1
2008-12,1
2009-01,4
2025-11,9
2025-12,4305003
2026-01,3724894
2026-02,3399866
2026-03,3952443
2026-04,3831230
2026-05,1


In [0]:
spark.sql("""
SELECT
    tpep_pickup_datetime,
    VendorID,
    PULocationID,
    DOLocationID
FROM nyctaxi_workspace.nyctaxi_01_bronze.yellow_trips_raw
WHERE tpep_pickup_datetime < '2025-12-01'
   OR tpep_pickup_datetime >= '2026-05-01'
ORDER BY tpep_pickup_datetime
""").display()

tpep_pickup_datetime,VendorID,PULocationID,DOLocationID
2001-01-01T09:23:58.000,2,138,263
2008-12-31T23:03:20.000,2,48,1
2009-01-01T00:02:17.000,2,48,48
2009-01-01T00:02:29.000,2,45,113
2009-01-01T00:03:23.000,2,230,142
2009-01-01T11:15:15.000,2,236,239
2025-11-30T20:48:56.000,2,233,162
2025-11-30T21:29:10.000,2,229,162
2025-11-30T21:49:06.000,2,229,233
2025-11-30T23:53:25.000,2,43,143


In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(tpep_pickup_datetime) AS min_pickup_date,
    MAX(tpep_pickup_datetime) AS max_pickup_date
FROM nyctaxi_workspace.nyctaxi_01_bronze.yellow_trips_raw
""").display()

total_rows,min_pickup_date,max_pickup_date
19213452,2001-01-01T09:23:58.000,2026-05-01T00:01:28.000
